# J1939 — truk, bus, dan alat berat

Alat diagnostik dan ECU mesin truk simulasi berbagi bus CAN virtual 250 kbit/s. Ganti bus dengan `await CanBus.OpenAsync("socketcan:can0")` (atau `slcan:`, `gsusb:`, `pcan:usb1`) untuk mendengarkan kendaraan sungguhan — hanya kendaraan yang Anda berwenang tangani.

## Persiapan
> Bekerja dari hasil clone? Jalankan `dotnet pack -c Release -o artifacts/packages` di root repo lalu tambahkan
> `#i "nuget: <repo>/artifacts/packages"` sebelum baris `#r`.

In [ ]:
#r "nuget: IoTCom.Net, 0.17.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.17.0-preview.1"

## Identifier 29-bit
Prioritas, parameter group number (PGN), dan alamat sumber; PGN PDU1 (PF < 240) juga membawa tujuan.

In [ ]:
using IoTCom.Net.Protocols.J1939;
using IoTCom.Net.Transport.Can;

var eec1Id = J1939Id.FromCanId(0x0CF00400);
Console.WriteLine($"priority {eec1Id.Priority}, PGN {eec1Id.Pgn} ({Pgn.Name(eec1Id.Pgn)}), source 0x{eec1Id.Source:X2}");
var reqId = J1939Id.FromCanId(0x18EA00F9);
Console.WriteLine($"PGN {reqId.Pgn} ({Pgn.Name(reqId.Pgn)}) from 0x{reqId.Source:X2} to 0x{reqId.Destination:X2}");

## SPN: skala, offset, dan 'tidak tersedia'
Setiap parameter punya posisi, resolusi, dan offset tetap. 0xFF… berarti tidak tersedia.

In [ ]:
foreach (var v in J1939Spn.Decode(Pgn.Eec1, Convert.FromHexString("F17D82A02200FF7D")))
    Console.WriteLine(v);
var et1 = new byte[8];
et1.AsSpan().Fill(0xFF);
J1939Spn.Encode(Pgn.Et1, 110, 88, et1);
Console.WriteLine($"ET1 with coolant 88 °C: {Convert.ToHexString(et1)}");

## Node, mesin, dan VIN lewat RTS/CTS
Alat mengklaim alamat 0xF9; VIN lebih dari 8 byte, sehingga mesin menjawab dengan transport protocol.

In [ ]:
var jNet = new VirtualCanNetwork("notebook-j1939");
var truck = J1939EngineSimulator.Create(jNet.CreateNode());
truck.Throttle = 50;
var tool = J1939Node.Create(jNet.CreateNode(), o => o.ReadOnly = true);
await tool.StartAsync();
await truck.StartAsync();
var vinMsg = await tool.RequestAsync(Pgn.VehicleIdentification, 0x00);
Console.WriteLine($"VIN: {System.Text.Encoding.ASCII.GetString(vinMsg.Data)} ({vinMsg.Data.Length} bytes)");
foreach (var (addr, name) in tool.Claims) Console.WriteLine($"0x{addr:X2}: function {name.Function}, identity {name.IdentityNumber}");

## Siaran dan DM1
Mesin menyiarkan EEC1/CCVS1 tiap 100 ms dan DM1 tiap detik. Mulai kebocoran oli dan lihat lampu kuning.

In [ ]:
J1939Dm1? lastDm1 = null;
double rpmNow = 0;
tool.MessageReceived += m =>
{
    if (m.Pgn == Pgn.Dm1) lastDm1 = J1939Dm1.Parse(m.Data);
    if (m.Pgn == Pgn.Eec1) rpmNow = m.Values.First(v => v.Spn == 190).Value ?? 0;
};
truck.OilLeak();
for (var i = 0; i < 40; i++) truck.Step(1);
await Task.Delay(1300);
Console.WriteLine($"{rpmNow:0} rpm, amber lamp {lastDm1?.AmberWarningLamp}");
foreach (var d in lastDm1!.Dtcs) Console.WriteLine($"SPN {d.Spn} {J1939Spn.Name(d.Spn)}: FMI {d.Fmi} {d.FailureMode}");

## Di jalur

In [ ]:
foreach (var f in J1939Spn.Describe(new CanFrame(0x18FECA00, Convert.FromHexString("04FF640001010000"), CanFrameFlags.Extended)))
    Console.WriteLine($"{f.Name,-10} {f.Value}");
await tool.DisposeAsync();
await truck.DisposeAsync();

## Lebih lanjut
`iotcom j1939 monitor --can sim`, `iotcom j1939 request vin --to 00 --can sim`, `iotcom j1939 claims --can sim`, *Panel truk lewat J1939* di Gallery, dan sampel J1939Monitor. Lihat `docs/id/protocols/j1939.md`.

*IoTCom.Net — dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*